In [1]:
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
import os

# get sql password
load_dotenv()
mysql_pw = os.getenv("MYSQL_PW")

db = mysql.connector.connect(
  host ="localhost",
  user ="root",
  passwd =mysql_pw,
  database ="proofbyindrag"
)

# tables = ['div', 'recurrence', 'sum', 'v1']
# prepare cursor for executing queries
conn = db.cursor()

In [23]:
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
import os
import warnings

# Suppress SyntaxWarning and UserWarning
warnings.filterwarnings("ignore", category=SyntaxWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# get sql password
load_dotenv()
mysql_pw = os.getenv("MYSQL_PW")
db = mysql.connector.connect(
  host ="localhost",
  user ="root",
  passwd =mysql_pw,
  database ="proofbyindrag"
)

# tables = ['div', 'recurrence', 'sum', 'v1']
# prepare cursor for executing queries
conn = db.cursor()

div_query = '''
-- view data
select *
FROM `div`;

-- Begin by creating all necessary columns
ALTER TABLE `div`
	ADD COLUMN total_score INT,
    ADD COLUMN quality_score VARCHAR(20),
    ADD COLUMN weaknesses LONGTEXT,
    ADD COLUMN strengths LONGTEXT,
    ADD COLUMN question LONGTEXT;

UPDATE `div`
    SET question =
    r"Use (strong) induction to prove the following claim: For any natural number \(n\), \(2n^3 + 3n^2 + n\) is divisible by 6.";

UPDATE `div`
SET Total_score =
	`Identify.Base.Case` +
	`Prove.Base.Case` +
	`Hypothesis.is.stated` +
	`Hypothesis.is.given.some.bound` +
    `Goal.is.Clear` +
    `Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k` +
    `Inductive.Hypothesis.is.applied`;

UPDATE `div`
    SET Quality_score =
        CASE
            WHEN Total_score <= 5 THEN 'Very Poor'
            WHEN Total_score <= 9 THEN 'Poor'
            WHEN Total_score <= 11 THEN 'OK'
            WHEN Total_score <= 14 THEN 'Excellent'
        END;

UPDATE `div`
SET Weaknesses = CONCAT(
    CASE `Identify.Base.Case`
        WHEN 0 THEN 'Failure to identify the base case. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Prove.Base.Case`
        WHEN 0 THEN 'Base case is not proven. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Hypothesis.is.stated`
        WHEN 0 THEN 'Failure to state the inductive hypothesis. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Hypothesis.is.given.some.bound`
        WHEN 0 THEN 'The bounds of the hypothesis are not stated. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Goal.is.Clear`
        WHEN 0 THEN 'The goal is not implicitly or explicitly stated. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k`
        WHEN 0 THEN 'The expression at k + 1 is not decomposed into the expression of size k. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Inductive.Hypothesis.is.applied`
        WHEN 0 THEN 'Inductive hypothesis is not applied at all. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END);

UPDATE `div`
	SET Strengths = CONCAT(
    CASE WHEN `Identify.Base.Case` = 2 THEN 'Successful identification of the base case. ' ELSE '' END,
    CASE WHEN `Prove.Base.Case` = 2 THEN 'Successfully proven base case. ' ELSE '' END,
    CASE WHEN `Hypothesis.is.stated` = 2 THEN 'Inductive hypothesis is explicitly or implicitly stated. ' ELSE '' END,
    CASE WHEN `Hypothesis.is.given.some.bound` = 2 THEN 'Successfully assigns correct bounds to the hypothesis. ' ELSE '' END,
    CASE WHEN `Goal.is.Clear` = 2 THEN 'Goal is implicitly or explicitly stated. ' ELSE '' END,
    CASE WHEN `Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k` = 2
         THEN 'Successfully decomposes the expression at k + 1 into the expression at k. ' ELSE '' END,
    CASE WHEN `Inductive.Hypothesis.is.applied` = 2
         THEN 'Successful application of the inductive hypothesis. ' ELSE '' END);

ALTER TABLE `div`
    RENAME COLUMN weaknesses TO Weaknesses,
    RENAME COLUMN strengths TO Strengths,
    RENAME COLUMN total_score to Total_score,
    RENAME COLUMN quality_score TO Quality_score,
    RENAME COLUMN question TO Question;'''

div_df = pd.read_sql(div_query, db)

folder_path = "Clean data"
div_file_name = "Div.json"
div_path = os.path.join(folder_path, div_file_name)

if os.path.exists(div_path):
    os.remove(div_path)

div_data = div_df.to_json(orient='records', indent=2)
print(div_data)

with open(div_path, "w", encoding="utf-8") as file:
    file.write(div_data)


[
  {
    "Proof":"Let the base case be for n = 0: \n\nAt n = 2, 2n^3 + 3n^2 + n = 0 which is divisible by 6\n\nInductive hypothesis: Suppose for all n that is represented by integers k ranging from 0 to k, 2n^3 + 3n^2 + n is divisble by 6\n\nInductive step: prove that 2(k + 1)^3 + 3(k + 1)^2 + k\n\nexpand the the factorial which reults in a polynomial with all of the k's being intgers and the coefficients being factors of 6. Thus, the whole expression is divisible by 6 due to transitivity.",
    "Identify.Base.Case":2,
    "Prove.Base.Case":2,
    "Hypothesis.is.stated":2,
    "Hypothesis.is.given.some.bound":2,
    "Goal.is.Clear":1,
    "Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k":0,
    "Inductive.Hypothesis.is.applied":0,
    "Total_score":9,
    "Quality_score":"Poor",
    "Weaknesses":"Partially correct but unclear or incomplete. The expression at k + 1 is not decomposed into the expression of size k. Inductive hypothesis is not applied at all. ",
    "Streng

In [46]:
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
import os
import warnings

# Suppress SyntaxWarning and UserWarning
warnings.filterwarnings("ignore", category=SyntaxWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# get sql password
load_dotenv()
mysql_pw = os.getenv("MYSQL_PW")
db = mysql.connector.connect(
  host ="localhost",
  user ="root",
  passwd =mysql_pw,
  database ="proofbyindrag"
)

# tables = ['div', 'recurrence', 'sum', 'v1']
# prepare cursor for executing queries
conn = db.cursor()

rec_query = '''
-- view data
select *
FROM proofbyindrag.recurrence;

-- Begin by creating all necessary columns
-- ALTER TABLE proofbyindrag.recurrence
  --  ADD COLUMN total_score INT,
  --  ADD COLUMN quality_score VARCHAR(20),
  --  ADD COLUMN weaknesses LONGTEXT,
  --  ADD COLUMN strengths LONGTEXT,
  --  ADD COLUMN question LONGTEXT;

ALTER TABLE proofbyindrag.recurrence
	MODIFY COLUMN weaknesses LONGTEXT,
    MODIFY COLUMN strengths LONGTEXT;

UPDATE proofbyindrag.recurrence
    SET question =
    'Suppose that g : N → R is defined by:
g(0) = 0
g(1) = 4/3
g(n) = (4/3)g(n − 1) − (1/3)g(n − 2), for n ≥ 2.

Use induction to prove that
g(n) = 2 − 2/(3^n) for every natural number n.
'
;

UPDATE proofbyindrag.recurrence
SET Total_score =
	`Identify.Base.Case` +
	`Prove.Base.Case` +
	`Hypothesis.is.stated` +
	`Hypothesis.is.given.some.bound` +
    `Goal.is.Clear` +
    `Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k` +
    `Inductive.Hypothesis.is.applied`;

UPDATE proofbyindrag.recurrence
    SET Quality_score =
        CASE
            WHEN Total_score <= 5 THEN 'Very Poor'
            WHEN Total_score <= 9 THEN 'Poor'
            WHEN Total_score <= 11 THEN 'OK'
            WHEN Total_score <= 14 THEN 'Excellent'
        END
;

UPDATE proofbyindrag.recurrence
SET Weaknesses = CONCAT(
    CASE `Identify.Base.Case`
        WHEN 0 THEN 'Failure to identify the base case. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Prove.Base.Case`
        WHEN 0 THEN 'Base case is not proven. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Hypothesis.is.stated`
        WHEN 0 THEN 'Failure to state the inductive hypothesis. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Hypothesis.is.given.some.bound`
        WHEN 0 THEN 'The bounds of the hypothesis are not stated. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Goal.is.Clear`
        WHEN 0 THEN 'The goal is not implicitly or explicitly stated. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k`
        WHEN 0 THEN 'The expression at k + 1 is not decomposed into the expression of size k. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Inductive.Hypothesis.is.applied`
        WHEN 0 THEN 'Inductive hypothesis is not applied at all. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END
);

UPDATE proofbyindrag.recurrence
	SET Strengths = CONCAT(
    CASE WHEN `Identify.Base.Case` = 2 THEN 'Successful identification of the base case. ' ELSE '' END,
    CASE WHEN `Prove.Base.Case` = 2 THEN 'Successfully proven base case. ' ELSE '' END,
    CASE WHEN `Hypothesis.is.stated` = 2 THEN 'Inductive hypothesis is explicitly or implicitly stated. ' ELSE '' END,
    CASE WHEN `Hypothesis.is.given.some.bound` = 2 THEN 'Successfully assigns correct bounds to the hypothesis. ' ELSE '' END,
    CASE WHEN `Goal.is.Clear` = 2 THEN 'Goal is implicitly or explicitly stated. ' ELSE '' END,
    CASE WHEN `Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k` = 2
         THEN 'Successfully decomposes the expression at k + 1 into the expression at k. ' ELSE '' END,
    CASE WHEN `Inductive.Hypothesis.is.applied` = 2
         THEN 'Successful application of the inductive hypothesis. ' ELSE '' END
);

ALTER TABLE proofbyindrag.recurrence
	DROP COLUMN Score_Feedback;

ALTER TABLE proofbyindrag.recurrence
RENAME COLUMN weaknesses TO Weaknesses,
RENAME COLUMN strengths TO Strengths,
RENAME COLUMN total_score to Total_score,
RENAME COLUMN quality_score TO Quality_score,
RENAME COLUMN question TO Question;'''

rec_df = pd.read_sql(rec_query, db)

rec_file_name = "Recurrence.json"
rec_path = os.path.join(folder_path, rec_file_name)

if os.path.exists(rec_path):
    os.remove(rec_path)

rec_data = rec_df.to_json(orient='records', indent=2)
print(rec_data)

#with open(rec_path, "w", encoding="utf-8") as file:
    #file.write(rec_data)

[
  {
    "Proof":"let n be a natural number,\nsince g(n) = g(n-1) + 1\/3g(n-1) -1\/3g(n-2), g(n-1)= g(n-2) + 1\/3g(n-2) - 1\/3g(n-3). Thus, g(n) = 1\/3g(n-1) + g(n - 2) -1\/3g(n-3).Following this sequence,  g(n) = 1\/3g(n-1) + 1\/3g(n-2) + ... + g(1) - 1\/3g(0). Since we can irrtate this fuction once agian to get 1\/3(g(n-1))= 1\/9g(n-2) + 1\/9g(n-3)+...+ 1\/3g(1) - 1\/9g(0).\nSo, the sum of 1\/(3^n)*(4\/3)*n. It is equals to(2(3^n-1))\/(3^n)\nThen, g(n) = 2 -2\/3n.\nQED",
    "Identify.Base.Case":0,
    "Prove.Base.Case":0,
    "Hypothesis.is.stated":0,
    "Hypothesis.is.given.some.bound":0,
    "Goal.is.Clear":0,
    "Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k":0,
    "Inductive.Hypothesis.is.applied":0,
    "Total_score":0,
    "Quality_score":"Very Poor",
    "weaknesses":"Failure to identify the base case. Base case is not proven. Failure to state the inductive hypothesis. The bounds of the hypothesis are not stated. The goal is not implicitly or explicitly s

In [43]:
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
import os
import warnings

warnings.filterwarnings("ignore", category=SyntaxWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# get sql password
load_dotenv()
mysql_pw = os.getenv("MYSQL_PW")
db = mysql.connector.connect(
  host ="localhost",
  user ="root",
  passwd =mysql_pw,
  database ="proofbyindrag"
)

# tables = ['div', 'recurrence', 'sum', 'v1']
# prepare cursor for executing queries
conn = db.cursor()

sum_query = '''-- view data
select *
FROM proofbyindrag.sum;

-- Begin by creating all necessary columns
ALTER TABLE proofbyindrag.sum
	ADD COLUMN total_score INT,
    ADD COLUMN quality_score VARCHAR(20),
    ADD COLUMN weaknesses LONGTEXT,
    ADD COLUMN strengths LONGTEXT,
    ADD COLUMN question LONGTEXT;

UPDATE proofbyindrag.sum
    SET question =
    'Use strong induction to prove the following claim:

Claim:

For all natural numbers n,

0·0! + 1·1! + 2·2! + ... + n·n! = (n + 1)! − 1.

Recall that 0! = 1.'
;

UPDATE proofbyindrag.sum
SET Total_score =
	`Identify.Base.Case` +
	`Prove.Base.Case` +
	`Hypothesis.is.stated` +
	`Hypothesis.is.given.some.bound` +
    `Goal.is.Clear` +
    `Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k` +
    `Inductive.Hypothesis.is.applied`;

UPDATE proofbyindrag.sum
    SET Quality_score =
        CASE
            WHEN Total_score <= 5 THEN 'Very Poor'
            WHEN Total_score <= 9 THEN 'Poor'
            WHEN Total_score <= 11 THEN 'OK'
            WHEN Total_score <= 14 THEN 'Excellent'
        END
;

UPDATE proofbyindrag.sum
SET Weaknesses = CONCAT(
    CASE `Identify.Base.Case`
        WHEN 0 THEN 'Failure to identify the base case. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Prove.Base.Case`
        WHEN 0 THEN 'Base case is not proven. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Hypothesis.is.stated`
        WHEN 0 THEN 'Failure to state the inductive hypothesis. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Hypothesis.is.given.some.bound`
        WHEN 0 THEN 'The bounds of the hypothesis are not stated. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Goal.is.Clear`
        WHEN 0 THEN 'The goal is not implicitly or explicitly stated. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k`
        WHEN 0 THEN 'The expression at k + 1 is not decomposed into the expression of size k. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Inductive.Hypothesis.is.applied`
        WHEN 0 THEN 'Inductive hypothesis is not applied at all. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END
);

UPDATE proofbyindrag.sum
	SET Strengths = CONCAT(
    CASE WHEN `Identify.Base.Case` = 2 THEN 'Successful identification of the base case. ' ELSE '' END,
    CASE WHEN `Prove.Base.Case` = 2 THEN 'Successfully proven base case. ' ELSE '' END,
    CASE WHEN `Hypothesis.is.stated` = 2 THEN 'Inductive hypothesis is explicitly or implicitly stated. ' ELSE '' END,
    CASE WHEN `Hypothesis.is.given.some.bound` = 2 THEN 'Successfully assigns correct bounds to the hypothesis. ' ELSE '' END,
    CASE WHEN `Goal.is.Clear` = 2 THEN 'Goal is implicitly or explicitly stated. ' ELSE '' END,
    CASE WHEN `Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k` = 2
         THEN 'Successfully decomposes the expression at k + 1 into the expression at k. ' ELSE '' END,
    CASE WHEN `Inductive.Hypothesis.is.applied` = 2
         THEN 'Successful application of the inductive hypothesis. ' ELSE '' END
);

ALTER TABLE proofbyindrag.sum
RENAME COLUMN weaknesses TO Weaknesses,
RENAME COLUMN strengths TO Strengths,
RENAME COLUMN total_score to Total_score,
RENAME COLUMN quality_score TO Quality_score,
RENAME COLUMN question TO Question;'''

sum_df = pd.read_sql(sum_query, db)

sum_file_name = "Sum.json"
sum_path = os.path.join(folder_path, sum_file_name)

if os.path.exists(sum_path):
    os.remove(sum_path)

sum_data = sum_df.to_json(orient='records', indent=2)
print(sum_data)

with open(sum_path, "w", encoding="utf-8") as file:
    file.write(sum_data)

[
  {
    "Proof":"Let $n$ be some arbitrary natural number.\n\nBase case: $\\sum_{p=0}^{0}(p*p!)=(0*1)=0$ which is equal to $(0+1)!-1=(1!)-1=0$\n\nInductive hypothesis: For some $k$, $\\sum_{p=0}^{n}(p*p!)=(n+1)!-1$ where $n=0...k$.\n\nInductive step: Show that $\\sum_{p=0}^{k+1}(p*p!)=((k+1)+1)!-1$.",
    "Identify.Base.Case":2,
    "Prove.Base.Case":2,
    "Hypothesis.is.stated":2,
    "Hypothesis.is.given.some.bound":2,
    "Goal.is.Clear":2,
    "Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k":0,
    "Inductive.Hypothesis.is.applied":0,
    "Total_score":10,
    "Quality_score":"OK",
    "Weaknesses":"The expression at k + 1 is not decomposed into the expression of size k. Inductive hypothesis is not applied at all. ",
    "Strengths":"Successful identification of the base case. Successfully proven base case. Inductive hypothesis is explicitly or implicitly stated. Successfully assigns correct bounds to the hypothesis. Goal is implicitly or explicitly stated. ",
  

In [44]:
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
import os
import warnings

# Suppress SyntaxWarning and UserWarning
warnings.filterwarnings("ignore", category=SyntaxWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# get sql password
load_dotenv()
mysql_pw = os.getenv("MYSQL_PW")
db = mysql.connector.connect(
  host ="localhost",
  user ="root",
  passwd =mysql_pw,
  database ="proofbyindrag"
)

# tables = ['div', 'recurrence', 'sum', 'v1']
# prepare cursor for executing queries
conn = db.cursor()

v1_query = '''select *
FROM proofbyindrag.v1;

-- Begin by creating all necessary columns
ALTER TABLE proofbyindrag.v1
	ADD COLUMN total_score INT,
    ADD COLUMN quality_score VARCHAR(20),
    ADD COLUMN weaknesses LONGTEXT,
    ADD COLUMN strengths LONGTEXT,
    ADD COLUMN question LONGTEXT;

UPDATE proofbyindrag.v1
    SET question =
    'For every natural number n,
0 + 1 + 2 + ... + n = n(n + 1) / 2.'
;

UPDATE proofbyindrag.v1
SET Total_score =
	`Identify.Base.Case` +
	`Prove.Base.Case` +
	`Hypothesis.is.stated` +
	`Hypothesis.is.given.some.bound` +
    `Goal.is.Clear` +
    `Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k` +
    `Inductive.Hypothesis.is.applied`;

UPDATE proofbyindrag.v1
    SET Quality_score =
        CASE
            WHEN Total_score <= 5 THEN 'Very Poor'
            WHEN Total_score <= 9 THEN 'Poor'
            WHEN Total_score <= 11 THEN 'OK'
            WHEN Total_score <= 14 THEN 'Excellent'
        END
;

UPDATE proofbyindrag.v1
SET Weaknesses = CONCAT(
    CASE `Identify.Base.Case`
        WHEN 0 THEN 'Failure to identify the base case. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Prove.Base.Case`
        WHEN 0 THEN 'Base case is not proven. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Hypothesis.is.stated`
        WHEN 0 THEN 'Failure to state the inductive hypothesis. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Hypothesis.is.given.some.bound`
        WHEN 0 THEN 'The bounds of the hypothesis are not stated. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Goal.is.Clear`
        WHEN 0 THEN 'The goal is not implicitly or explicitly stated. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k`
        WHEN 0 THEN 'The expression at k + 1 is not decomposed into the expression of size k. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END,
    CASE `Inductive.Hypothesis.is.applied`
        WHEN 0 THEN 'Inductive hypothesis is not applied at all. '
        WHEN 1 THEN 'Partially correct but unclear or incomplete. '
        ELSE ''
    END
);

UPDATE proofbyindrag.v1
	SET Strengths = CONCAT(
    CASE WHEN `Identify.Base.Case` = 2 THEN 'Successful identification of the base case. ' ELSE '' END,
    CASE WHEN `Prove.Base.Case` = 2 THEN 'Successfully proven base case. ' ELSE '' END,
    CASE WHEN `Hypothesis.is.stated` = 2 THEN 'Inductive hypothesis is explicitly or implicitly stated. ' ELSE '' END,
    CASE WHEN `Hypothesis.is.given.some.bound` = 2 THEN 'Successfully assigns correct bounds to the hypothesis. ' ELSE '' END,
    CASE WHEN `Goal.is.Clear` = 2 THEN 'Goal is implicitly or explicitly stated. ' ELSE '' END,
    CASE WHEN `Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k` = 2
         THEN 'Successfully decomposes the expression at k + 1 into the expression at k. ' ELSE '' END,
    CASE WHEN `Inductive.Hypothesis.is.applied` = 2
         THEN 'Successful application of the inductive hypothesis. ' ELSE '' END
);

ALTER TABLE proofbyindrag.V1
	RENAME COLUMN weaknesses TO Weaknesses,
	RENAME COLUMN strengths TO Strengths,
	RENAME COLUMN total_score to Total_score,
	RENAME COLUMN quality_score TO Quality_score,
	RENAME COLUMN question TO Question;'''

v1_df = pd.read_sql(v1_query, db)

v1_file_name = "v1.json"
v1_path = os.path.join(folder_path, v1_file_name)

if os.path.exists(v1_path):
    os.remove(v1_path)

v1_data = v1_df.to_json(orient='records', indent=2)
print(v1_data)

with open(v1_path, "w", encoding="utf-8") as file:
    file.write(v1_data)

[
  {
    "Proof":"Let us take an arbitrary natural number n\n\n1)For n = 1\n\n$ \\sum_{i=0}^{1}$ = 1\nn(n+1)\/2 = 1\nso, the statement holds true\n\n2)Assuming statement holds true for some n = k\n  $ \\sum_{i=0}^{k}$ = k(k+1)\/2\n  \n3)We need to prove the statement holds for k + 1 =>   $ \\sum_{i=0}^{k+1}$ = (k+2)(k+1)\/2\n\n$ \\sum_{i=0}^{k+1}$ = (1 + 2 + 3 + ........ + k) + k + 1\n$ \\sum_{i=0}^{k+1}$ = k(k+1)\/2 + k+1 (using the above assumption)\n$ \\sum_{i=0}^{k+1}$ = (k+1)(k\/2  + 1) = (k+1)(k+2)\/2\nhence proved.\n\nhence we proved the above statement by induction",
    "Identify.Base.Case":2,
    "Prove.Base.Case":2,
    "Hypothesis.is.stated":1,
    "Hypothesis.is.given.some.bound":2,
    "Goal.is.Clear":1,
    "Expression.of.Size.k.1.is.decomposed.into.expression.of.size.k":2,
    "Inductive.Hypothesis.is.applied":2,
    "Total_score":12,
    "Quality_score":"Excellent",
    "Weaknesses":"Partially correct but unclear or incomplete. Partially correct but unclear or incompl